# Bombay Was Seven Islands

The 1843 map with today's coastline bent to fit over it. Run `fetch_week1.py day6` first.

## 1 · Settings

In [ ]:
DAY      = 6
PROMPT   = "Vintage"
SLUG     = "vintage-bombay-islands"
NOTEBOOK = "day06_vintage_bombay_islands.ipynb"
TITLE    = "Bombay Was\nSeven Islands"
CREDITS  = ("Map prepared 1843 for Mr. Murphy, reproduced Poona 1894.\nBritish Library via Wikimedia Commons.\n"
            "Today's coastline: © OpenStreetMap contributors,\nfit by hand to 13 landmarks, so read it as approximate.")

## 2 · Load

In [ ]:
import geopandas as gpd, numpy as np
from PIL import Image
from scipy.interpolate import RBFInterpolator
from shapely.geometry import box
from mapkit import poster, publish
from mapkit.paths import PROCESSED

scan = Image.open(PROCESSED / "bombay_1843.jpg").convert("RGB")
coast = gpd.read_file(PROCESSED / "mumbai_coastline.geojson")

## 3 · Analysis, every number on the map comes from here

In [ ]:
# Landmarks on the old map (pixel x, y in our 1377 x 3000 copy) matched to where they are today (lon, lat)
LANDMARKS = {
    "Colaba tip":          (642, 2913, 72.8080, 18.8935),
    "Old Woman's Island":  (822, 2517, 72.8195, 18.9110),
    "Bombay Castle":       (900, 2232, 72.8385, 18.9285),
    "Fort George":         (895, 2100, 72.8400, 18.9405),
    "Malabar Point":       (195, 2292, 72.7945, 18.9470),
    "Mazagaon":            (885, 1695, 72.8440, 18.9650),
    "Mahalaxmi":           (245, 1605, 72.8065, 18.9775),
    "Haji Ali":            (275, 1448, 72.8090, 18.9825),
    "Sewri Fort":          (982, 1020, 72.8590, 19.0000),
    "Worli tip":           (165,  788, 72.8170, 19.0215),
    "Mahim Fort":          (570,  382, 72.8395, 19.0420),
    "Bandra tip":          (138,  518, 72.8190, 19.0420),
    "Sion (Riwa)":         (783,  285, 72.8650, 19.0465),
}
names = list(LANDMARKS)
pts = np.array([LANDMARKS[n] for n in names], float)
px, ll = pts[:, :2], pts[:, 2:]

# A thin plate spline bends today's coastline to pass close to every landmark, leaving the old map untouched
warp = RBFInterpolator(ll, px, kernel="thin_plate_spline", smoothing=1e-4)
check = warp(ll)
residual = np.hypot(*(check - px).T)
print("Fit error at each landmark, in scan pixels (the scan is 3000 tall)")
for n, r in zip(names, residual):
    print(f"  {n:20} {r:6.1f}")
print(f"  RMS {np.sqrt((residual ** 2).mean()):.1f}")

# Only the island city. Far outside the landmarks a warp stops meaning anything.
keep = box(72.78, 18.885, 72.872, 19.058)
near = coast.clip(keep)

## 4 · Map

In [ ]:
paper = np.array(scan.resize((40, 40)))[1:3, 1:3].reshape(-1, 3).mean(0) / 255
fig, ax = poster.page(tuple(paper))
scan_w = 1350 / scan.height * scan.width
left = 1080 - scan_w
ax_scan = fig.add_axes([left / 1080, 0, scan_w / 1080, 1])
ax_scan.imshow(scan, interpolation="lanczos")
ax_scan.set_axis_off()
ax_scan.set_xlim(0, scan.width); ax_scan.set_ylim(scan.height, 0)
RED = "#c8102e"
for g in near.geometry:
    for line in (g.geoms if g.geom_type == "MultiLineString" else [g]):
        xy = warp(np.array(line.coords)[:, :2])
        ax_scan.plot(xy[:, 0], xy[:, 1], color=RED, lw=1.25, solid_capstyle="round", solid_joinstyle="round")
ax_scan.set_xlim(14, scan.width - 8); ax_scan.set_ylim(scan.height - 8, 8)

INKY, SOFT = "#2e2519", "#6d5f4a"
fig.text(0.05, 0.945, TITLE, color=INKY, fontproperties=poster.font(poster.OLD, 46), va="top", linespacing=1.02)
fig.text(0.05, 0.795, "Mumbai has never believed\nin coastlines.", color=RED, fontproperties=poster.font(poster.OLD_ITALIC, 22), va="top", linespacing=1.15)
fig.text(0.05, 0.68, "\n".join(["Mahim", "Worli", "Parel", "Mazagaon", "Bombay", "Old Woman's Island", "Colaba"]), color=INKY,
         fontproperties=poster.font(poster.OLD_ITALIC, 17), va="top", linespacing=1.55)
fig.text(0.05, 0.445, "The seven-island story traces back\nto this 1843 map, which was itself\na guess at the old landscape.",
         color=SOFT, fontproperties=poster.font(poster.SANS, 11.5), va="top", linespacing=1.45)
key = fig.add_axes([0.05, 0.325, 0.07, 0.03]); key.set_axis_off(); key.set_xlim(0, 1); key.set_ylim(0, 1)
key.plot([0, 1], [0.5, 0.5], color=RED, lw=2.6, solid_capstyle="round")
fig.text(0.13, 0.34, "Today's shoreline", color=RED, fontproperties=poster.font(poster.SANS_BOLD, 13), va="center")
fig.text(0.05, 0.022, CREDITS, color=SOFT, fontproperties=poster.font(poster.SANS, 7.4), va="bottom", linespacing=1.4)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle="The 1843 map, with today's coastline drawn over it", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)